In [1]:
# Import libraries
import pandas as pd
import numpy as np
import ast
import torch
import clip
from PIL import Image
import os
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Set device (GPU is better than CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using device: {device}')

c:\porsche-appreciation-predictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
# Load data
df = pd.read_csv('../data/csv/porsche_data.csv')

# Parse image paths
# Check how many listings have images, and how many
def parse_image_path(path):
    if pd.isna(path) or path == '':
        # If there's no image path for that listing, return an empty list
        return []
    
    if isinstance(path, str):
        # Check that image_path is a string, then convert it to a Python list
        return ast.literal_eval(path)
    if isinstance(path, list):
        return path

df['image_paths'] = df['image_paths'].apply(parse_image_path)

# Only look at listings with images/image file paths in the row
df = df[df['image_paths'].apply(len)>0]

print(f'Loaded {len(df)} listings with images')
print(f'Sample row: {df.iloc[0]["id"]}')

Loaded 1234 listings with images
Sample row: p00001


In [3]:
# Define models

# CLIP for images
print('\nLoading CLIP model ...')
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
print('CLIP model loaded!')

# SentenceTransformer for text
print('\nLoading SentenceTransformer model ...')
text_model = SentenceTransformer('all-MiniLM-L6-v2')
print('SentenceTransformer model loaded!')


Loading CLIP model ...
CLIP model loaded!

Loading SentenceTransformer model ...
SentenceTransformer model loaded!


In [4]:
# Initialise Qdrant (vector space)
# TEMPORARY memory
# client = QdrantClient(":memory:")
# store Qdrant in disk
client = QdrantClient(path="../data/qdrant_db")

# Create collection for IMAGE embeddings
image_collection_name = 'porsche_images'

try:
    client.create_collection(
        collection_name = image_collection_name,
        vectors_config =  VectorParams(size=512, distance=Distance.COSINE)
    )
    print(f'Created collection: {image_collection_name}')
except Exception as e:
    print(f'Collection might already exist: {e}')



# Create collection for TEXT embeddings
text_collection_name = 'porsche_text'

try:
    client.create_collection(
        collection_name = text_collection_name,
        vectors_config =  VectorParams(size=384, distance=Distance.COSINE)
    )
    print(f'Created collection: {text_collection_name}')
except Exception as e:
    print(f'Collection might already exist: {e}')

Created collection: porsche_images
Created collection: porsche_text


In [5]:
from pathlib import Path

# Go up 2 levels up the directory of image_path
workspace_root = Path.cwd().parent.parent

def resolve_path(csv_path):
    # Convert path in CSV to absolute path
    # For image path and text path
    if pd.isna(csv_path) or csv_path == '':
        return None
    if csv_path.startswith('project/'):
        resolved = csv_path.replace('project/', '../', 1)
    else:
        resolved = csv_path
    
    return os.path.normpath(resolved)
    # if os.path.isabs(csv_path):
    #     return csv_path
    # return str((workspace_root / csv_path).resolve())

In [6]:
# Function: Encode the images

def encode_image(image_path):
    # Returns embedding vector or None (if error)
    try:
        # Resolve image_path first
        full_path = resolve_path(image_path)
        if full_path is None or not os.path.exists(full_path):
            print(f'Image not found: {image_path}')
            return None

        image = Image.open(full_path).convert('RGB')
        image_tensor = clip_preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = clip_model.encode_image(image_tensor)
            # Normalise all features/cols so they are comparable
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            
        return image_features.cpu().numpy()[0]
    except Exception as e:
        print(f'Error encoding image {image_path}: {e}')
        return None



# Test on first image in first row of data:
# "[""project/data/images/p00001_image_1.jpg"", ""project/data/images/p00001_image_10.jpg"", ""project/data/images/p00001_image_11.jpg"", ""project/data/images/p00001_image_12.jpg"", ""project/data/images/p00001_image_2.jpg"", ""project/data/images/p00001_image_3.jpg"", ""project/data/images/p00001_image_4.jpg"", ""project/data/images/p00001_image_5.jpg"", ""project/data/images/p00001_image_6.jpg"", ""project/data/images/p00001_image_7.jpg"", ""project/data/images/p00001_image_8.jpg"", ""project/data/images/p00001_image_9.jpg""]"
sample_image_path = df.iloc[0]['image_paths'][0]
resolved_path = resolve_path(sample_image_path)

if resolved_path and os.path.exists(resolved_path):
    test_embedding = encode_image(sample_image_path)
    print(f'Test embedding shape: {test_embedding.shape}')
    if test_embedding is not None:
        print('Image encoding works!')
else:
    print(f'Sample image not found. Original: {sample_image_path}')
    print(f'Resolved path exists: {os.path.exists(resolved_path)}')
    print(f'Resolved path: {resolved_path}')
    print(f'Current working directory: {os.getcwd()}')

Test embedding shape: (512,)
Image encoding works!


In [7]:
# Encode text with SentenceTransformer
def encode_text(description_path):
    try:
        # Resolve the path first
        full_path = resolve_path(description_path)
        # If there is no path there
        if full_path is None or not os.path.exists(full_path):
            return None
        # if not os.path.exists(description_path):
        #     return None
        
        with open(full_path, 'r', encoding='utf-8') as f:
            text = f.read()
        # If contents of that text file is empty
        if not text.strip():
            return None
        
        # Now encode the text
        embedding = text_model.encode(text, normalize_embeddings=True)
        return embedding
    except Exception as e:
        print(f'Failed to encode text {full_path}: {e}')
        return None

# Test on one description
# project/data/descriptions/p00001_seller_description.txt
sample_desc_path = df.iloc[0]['seller_description'].replace('project/', '../', 1)

if sample_desc_path and os.path.exists(sample_desc_path):
    test_embedding_text = encode_text(sample_desc_path)
    print(f'Test embedding shape: {test_embedding_text.shape}')
    if test_embedding_text is not None:
        print('Text embedding is successful!')
else:
    print(f'Sample text not found. Original: {sample_desc_path}')
    print(f'Current working directory: {os.getcwd()}')

Test embedding shape: (384,)
Text embedding is successful!


In [8]:
# Generate embeddings (images + text) for all car listings

image_embeddings_list = []
text_embeddings_list = []
metadata_list = []
failed_indices = []

# Check image_paths data
print(f"Sample image_paths: {df.iloc[0]['image_paths']}")
print(f"Is it a list? {isinstance(df.iloc[0]['image_paths'], list)}")
print(f"Is it empty? {len(df.iloc[0]['image_paths']) == 0 if isinstance(df.iloc[0]['image_paths'], list) else 'not a list'}")

# DEBUG
failed_images = 0
failed_images_encode = 0
failed_texts = 0
failed_texts_encode = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing listings"):     # Cool progress bar
    listing_id = row['id']

    # Encode first listing image
    image_paths = row['image_paths']
    if image_paths:
        image_embedding = encode_image(image_paths[0])
        if image_embedding is None:
            # Note down which images failed to encode
            failed_image_encode += 1
            failed_indices.append(idx)
            continue
    else:
        # Note down if there are missing image paths
        failed_images += 1
        failed_indices.append(idx)
        continue

    # Encode listing description (text)
    text_path = row['seller_description']
    if text_path:
        text_embedding = encode_text(text_path)
        if text_embedding is None:
            # Note down whih texts failed to encode
            failed_texts_encode += 1
            failed_indices.append(idx)
            continue
    else:
        # Note down if any text path is missing
        failed_texts_encode += 1
        failed_indices.append(idx)
        continue



    # Store image and text embeddings in lists
    image_embeddings_list.append(image_embedding)
    text_embeddings_list.append(text_embedding)
    # Store metadata of each listings
    metadata_list.append({
        'listing_id' : listing_id,
        'model_year' : row['model_year'],
        'model_type' : row['model_type'],
        'mileage' : row['mileage'],
        'condition' : row['condition'],
        'price_now' : row['price_now'],
        'source' : row['source']
    })





print(f'\nProcessed {len(image_embeddings_list)}/{len(df)} rows')
print(f'Failed: {len(failed_indices)} listings')

print(f'\tNo. of images absent: {failed_images}')
print(f'\tNo. of images encoding failed: {failed_images_encode}')
print(f'\tNo. of texts absent: {failed_texts}')
print(f'\tNo. of texts encoding failed: {failed_texts_encode}')

Sample image_paths: ['project/data/images/p00001_image_1.jpg', 'project/data/images/p00001_image_10.jpg', 'project/data/images/p00001_image_11.jpg', 'project/data/images/p00001_image_12.jpg', 'project/data/images/p00001_image_2.jpg', 'project/data/images/p00001_image_3.jpg', 'project/data/images/p00001_image_4.jpg', 'project/data/images/p00001_image_5.jpg', 'project/data/images/p00001_image_6.jpg', 'project/data/images/p00001_image_7.jpg', 'project/data/images/p00001_image_8.jpg', 'project/data/images/p00001_image_9.jpg']
Is it a list? True
Is it empty? False


Processing listings: 100%|██████████| 1234/1234 [04:50<00:00,  4.25it/s]


Processed 1234/1234 rows
Failed: 0 listings
	No. of images absent: 0
	No. of images encoding failed: 0
	No. of texts absent: 0
	No. of texts encoding failed: 0


In [9]:
# Put each embedding grouped into 2 arrays, for image and text
image_embeddings_array = np.array(image_embeddings_list)
text_embeddings_array = np.array(text_embeddings_list)

# PUT INTO VECTOR SPACE
# image embeddings
image_points = [
    PointStruct(
        id=idx,
        # get vector from huge image list
        vector=image_embeddings_array[idx].tolist(),
        # info of each image; id, year, model type, etc.
        payload=metadata_list[idx]
    )
    for idx in range(len(image_embeddings_array))
]

client.upsert(
    collection_name=image_collection_name,
    points=image_points
)

print(f'Uploaded {len(image_points)} image embeddings to Qdrant')

# text embeddings
text_points = [
    PointStruct(
        id=idx,
        # get vector from huge text list
        vector=text_embeddings_array[idx].tolist(),
        # info of each text; id, year, model type, etc.
        payload=metadata_list[idx]
    )
    for idx in range(len(text_embeddings_array))
]

client.upsert(
    collection_name=text_collection_name,
    points=text_points
)

print(f'Uploaded {len(text_points)} text embeddings to Qdrant')

Uploaded 1234 image embeddings to Qdrant
Uploaded 1234 text embeddings to Qdrant


In [10]:
# TEST: similarity search
test_listing_idx = 0
test_image_embedding = image_embeddings_array[test_listing_idx]

# search for similar images
search_results = client.query_points(
    collection_name=image_collection_name,
    query=test_image_embedding.tolist(), 
    limit=5 
)

# # DEBUG: structure of output
# print(f'Type of search_results: {type(search_results)}')
# print(f'Attributes of search_results: {dir(search_results)}')
# print(f'\nFirst result: {search_results.points[0] if hasattr(search_results, "points") and search_results.points else "No points"}')
# print(f'Type of first result: {type(search_results.points[0] if hasattr(search_results, "points") and search_results.points else "No points")}')

print(f'Top 5 similar listings (by image):')
for result in search_results.points:
    print(f"\tScore: {result.score:.4f} | ID: {result.payload['listing_id']} | Model: {result.payload['model_type']} | Year: {result.payload['model_year']}")

Top 5 similar listings (by image):
	Score: 1.0000 | ID: p00001 | Model: 944 | Year: 1987.0
	Score: 0.8707 | ID: p00998 | Model: 928 | Year: 1988.0
	Score: 0.8427 | ID: p00448 | Model: 914 | Year: 1976.0
	Score: 0.8410 | ID: p00427 | Model: 944 | Year: 1988.0
	Score: 0.8392 | ID: p01181 | Model: 924 | Year: 1988.0


In [11]:
# Find all methods in Qdrant library that have 'search' or 'query' in the name
# print([method for method in dir(client) if 'search' in method.lower()])
# print([method for method in dir(client) if 'query' in method.lower()])